# 4-1. 단계가 이어진 계획

선택 심화. `4`를 마친 뒤 본다.

`4`의 Plan&Execute 는 단계가 서로 독립일 때만 맞았다. 이 노트북의 과제는 다음 단계의 검색어가 앞 단계의 결과로 정해진다. 미리 값을 알 수 없으니 단계를 독립으로 나눌 수 없다.
값 대신 자리표시자로 계획을 쓰는 ReWOO, 그 계획을 의존 그래프로 보고 동시에 실행하는 DAG, 그리고 계획을 모델 안에 맡기는 추론 effort 를 비교한다.

### 공통 설정

`4`와 같은 블록이다. 실행은 하위 모델로 하고 상위 모델은 계획과 기준선에만 쓴다.

In [1]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from typing import Literal
from pydantic import BaseModel

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()
EFFORT = {"effort": "low"}

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

도구는 `4`의 둘이다. `grep_docs`는 문서 제목을 그대로 받아도 되게 밑줄을 띄어쓰기로 바꿔 찾는다. 이 과제는 앞에서 고른 문서의 제목이 다음 검색어가 되기 때문이다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    pattern = pattern.replace("_", " ")
    hits = [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]
    return {"문서수": len(hits), "문서": hits}


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(title)
    return f.read_text(encoding="utf-8")[:2000]


def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("grep_docs", "낱말이 들어 있는 문서 수와, 문서별 제목과 등장 횟수를 돌려준다."
                           " 문서 제목을 넣으면 밑줄을 띄어쓰기로 바꿔 찾는다.",
              {"pattern": {"type": "string", "description": "찾을 낱말 또는 문서 제목"}}),
         spec("read_doc", "문서 한 편의 본문 앞부분을 돌려준다.",
              {"title": {"type": "string", "description": "문서 제목. grep_docs 가 준 것 그대로"}})]
REGISTRY = {"grep_docs": grep_docs, "read_doc": read_doc}


def call_tool(name, args):
    try:
        return json.dumps(REGISTRY[name](**args), ensure_ascii=False)
    except Exception as e:
        return f"error: {e}"

### 과제와 채점기

시작 낱말마다 문서 사슬을 세 번 따라간다. 낱말이 가장 많이 나오는 문서를 고르고, 그 문서의 제목으로 다시 검색해 다음 문서를 고른다. 이미 고른 문서는 뺀다.
마지막 문서는 읽어서 첫 연도를 뽑는다. 사슬 안에서는 한 단계도 앞당길 수 없고, 사슬끼리는 서로 상관이 없다.
정답은 같은 경로를 코드로 밟아 만든다.

In [3]:
STARTS = ["독립신문", "신민회", "도쿄", "이토 히로부미", "권업회", "홍범도", "총독부", "흥사단"]


def top_doc(pattern, exclude):
    hits = [d for d in grep_docs(pattern)["문서"] if d["title"] not in exclude]
    return max(hits, key=lambda d: d["count"])["title"]


FACTS = {}
for w in STARTS:
    chain, pattern = [], w
    for _ in range(3):
        chain.append(top_doc(pattern, chain))
        pattern = chain[-1]
    FACTS[w] = (*chain, re.search(r"(\d{4})년", read_doc(chain[-1]))[1])

for w, f in FACTS.items():
    print(f"{w:8s} {f}")

독립신문     ('서재필', '한국친우회', '이승만', '1875')
신민회      ('이회영', '신채호', '김규식', '1881')
도쿄       ('이봉창_의거', '한인애국단', '김구', '1876')
이토 히로부미  ('안중근', '안공근', '김구', '1876')
권업회      ('권업회', '이상설', '이회영', '1867')
홍범도      ('자유시_참변', '대한민국_임시정부', '이승만', '1875')
총독부      ('안재홍', '김규식', '이승만', '1875')
흥사단      ('안창호', '대한민국_임시정부', '이승만', '1875')


사슬 끝이 김구나 이승만으로 모이는 낱말은 모델이 외운 지식으로 맞힐 수 있어서 덜 골랐다. 그래도 몇 개는 남아 있다.

In [4]:
TASK = ("아래 시작 낱말마다 문서 사슬을 따라간다.\n"
        "1차: 시작 낱말이 가장 많이 나오는 문서\n"
        "2차: 1차 문서의 제목으로 검색해, 이미 고른 문서를 뺀 것 중 가장 많이 나오는 문서\n"
        "3차: 2차 문서의 제목으로 검색해, 이미 고른 문서를 뺀 것 중 가장 많이 나오는 문서\n"
        "연도: 3차 문서를 read_doc 으로 읽어 본문에 처음 나오는 연도\n"
        "'이미 고른 문서'는 그 사슬에서 앞서 고른 문서다. 짐작하지 말고 도구로 확인한다.\n"
        + "\n".join(f"- {w}" for w in STARTS)
        + "\n마지막에 | 시작 | 1차 | 2차 | 3차 | 연도 | 다섯 칸짜리 마크다운 표 하나로 정리한다.")
print(TASK)

아래 시작 낱말마다 문서 사슬을 따라간다.
1차: 시작 낱말이 가장 많이 나오는 문서
2차: 1차 문서의 제목으로 검색해, 이미 고른 문서를 뺀 것 중 가장 많이 나오는 문서
3차: 2차 문서의 제목으로 검색해, 이미 고른 문서를 뺀 것 중 가장 많이 나오는 문서
연도: 3차 문서를 read_doc 으로 읽어 본문에 처음 나오는 연도
'이미 고른 문서'는 그 사슬에서 앞서 고른 문서다. 짐작하지 말고 도구로 확인한다.
- 독립신문
- 신민회
- 도쿄
- 이토 히로부미
- 권업회
- 홍범도
- 총독부
- 흥사단
마지막에 | 시작 | 1차 | 2차 | 3차 | 연도 | 다섯 칸짜리 마크다운 표 하나로 정리한다.


`4`처럼 마지막 표만 채점한다. 같은 제목이 여러 칸에 나올 수 있으므로 칸 위치로 대조한다. 제목의 밑줄과 띄어쓰기는 같게 본다.

In [5]:
def last_table(text):
    blocks, cur = [], []
    for line in text.splitlines():
        if line.count("|") >= 3:
            cur.append(line)
        elif cur:
            blocks.append(cur)
            cur = []
    if cur:
        blocks.append(cur)
    return blocks[-1] if blocks else []


def norm(s):
    return s.strip().strip("`*").replace(" ", "_")


def grade(text):
    rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in last_table(text or "")]
    hit = {}
    for w, facts in FACTS.items():
        row = next((r for r in rows if r[0].strip("`* ") == w), [])
        for i, name in enumerate(["1차", "2차", "3차"]):
            hit[f"{w}/{name}"] = len(row) > i + 1 and norm(row[i + 1]) == facts[i]
        hit[f"{w}/연도"] = len(row) > 4 and facts[3] in row[4]
    return sum(hit.values()), hit


FULL = len(STARTS) * 4
print(FULL, "개 항목")

32 개 항목


점수와 비용을 모으는 함수는 `4`와 같다.

In [6]:
def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total


def trial(label, fn, n=3):
    scores, mark, t0 = [], len(USAGE), time.perf_counter()
    for i in range(n):
        text = fn()
        score, hit = grade(text)
        scores.append(score)
        print(f"{label} {i + 1}회  {score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v][:6]}"
              + ("" if text else "  (답 없음, 반복 예산 소진)"))
    return {"점수": scores, "호출": len(USAGE) - mark, "비용": dollars(USAGE[mark:]),
            "초": time.perf_counter() - t0}


RESULTS = {}

### ReAct

`4`의 루프에서 앵커를 뺐고 effort 를 인자로 받게 했다. 사슬 여덟 개를 한 대화가 전부 들고 간다.

In [7]:
def run(task, model=LOWER, effort=EFFORT, max_turns=15):
    messages = [{"role": "user", "content": task}]
    for turn in range(1, max_turns + 1):
        r = keep(client.responses.create(model=model, input=messages, tools=TOOLS,
                                         reasoning=effort, store=False))
        calls = [i for i in r.output if i.type == "function_call"]
        if not calls:
            return r.output_text, turn, messages
        messages += [i.model_dump(exclude_none=True) for i in r.output]
        for c in calls:
            messages.append({"type": "function_call_output", "call_id": c.call_id,
                             "output": call_tool(c.name, json.loads(c.arguments))})
    return None, turn, messages

한 번 돌려 답과 점수를 본다. 도구 호출이 사슬 수 x 4 = 32번보다 적으면 어딘가를 건너뛴 것이다.

In [8]:
answer, turns, messages = run(TASK)
print("턴", turns, " 도구 호출", sum(m.get("type") == "function_call" for m in messages))
print(answer)
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

턴 7  도구 호출 25
| 시작 | 1차 | 2차 | 3차 | 연도 |
|---|---|---|---|---|
| 독립신문 | 서재필 | 한국친우회 | 대한민국_임시정부 | 1919 |
| 신민회 | 안창호 | 대한민국_임시정부 | 이승만 | 1875 |
| 도쿄 | 안재홍 | 이승만 | 안공근 | 1889 |
| 이토 히로부미 | 안중근 | 안공근 | 안중근 | 1879 |
| 권업회 | 권업회 | 이상설 | 이회영 | 1867 |
| 홍범도 | 안중근 | 안공근 | 안중근 | 1879 |
| 총독부 | 안창호 | 대한민국_임시정부 | 이승만 | 1875 |
| 흥사단 | 안창호 | 대한민국_임시정부 | 이승만 | 1875 |
14/32  못 채운 것 ['독립신문/3차', '독립신문/연도', '신민회/1차', '신민회/2차', '신민회/3차', '신민회/연도', '도쿄/1차', '도쿄/2차', '도쿄/3차', '도쿄/연도', '이토 히로부미/3차', '이토 히로부미/연도', '홍범도/1차', '홍범도/2차', '홍범도/3차', '홍범도/연도', '총독부/1차', '총독부/2차']


In [9]:
RESULTS["ReAct"] = trial("ReAct", lambda: run(TASK)[0])

ReAct 1회  24/32  못 채운 것 ['신민회/1차', '신민회/2차', '신민회/3차', '신민회/연도', '도쿄/1차', '도쿄/2차']
ReAct 2회  21/32  못 채운 것 ['독립신문/2차', '신민회/2차', '신민회/3차', '신민회/연도', '도쿄/1차', '도쿄/2차']
ReAct 3회  23/32  못 채운 것 ['신민회/1차', '신민회/2차', '신민회/3차', '신민회/연도', '도쿄/1차', '도쿄/2차']


### 계획을 모델 안에 맡긴다

추론 모델은 답하기 전에 스스로 계획을 세운다. effort 를 올리면 그 계획에 쓰는 토큰이 는다.
바깥에 계획을 만들지 않고 effort 만 올려도 되는지 본다.

In [10]:
RESULTS["ReAct(effort high)"] = trial("ReAct(effort high)", lambda: run(TASK, effort={"effort": "high"})[0])

ReAct(effort high) 1회  28/32  못 채운 것 ['도쿄/1차', '도쿄/2차', '도쿄/3차', '도쿄/연도']
ReAct(effort high) 2회  26/32  못 채운 것 ['도쿄/1차', '도쿄/2차', '도쿄/3차', '도쿄/연도', '총독부/1차', '총독부/2차']
ReAct(effort high) 3회  24/32  못 채운 것 ['독립신문/3차', '독립신문/연도', '도쿄/1차', '도쿄/2차', '도쿄/3차', '도쿄/연도']


기준선으로 상위 모델 ReAct 도 돌린다. effort 는 다른 갈래와 같은 low 다.

In [11]:
RESULTS["ReAct(상위)"] = trial("ReAct(상위)", lambda: run(TASK, model=MODEL)[0])

ReAct(상위) 1회  32/32  못 채운 것 []
ReAct(상위) 2회  32/32  못 채운 것 []
ReAct(상위) 3회  32/32  못 채운 것 []


### 자리표시자 계획 (ReWOO)

`4`의 계획자는 "1차 문서를 찾는다"까지는 쓸 수 있어도 그다음 검색어를 적을 수 없었다. 값을 모르기 때문이다.
ReWOO 는 값 대신 자리표시자를 쓴다. `#E1` 은 노드 E1 의 결과이고, 실행할 때 그 결과로 바뀐다. (placeholder로 처리)
계획자는 도구 결과를 한 번도 보지 않고 계획 전체를 한 번에 쓴다.
이름이 여기서 나왔다. ReWOO 는 **Reasoning WithOut Observation**, 곧 **관찰(도구 결과) 없이 추론**한다는 뜻이다(Xu 외, 2023).

노드의 도구는 셋이다. `grep_docs`와 `read_doc`은 코드가 그대로 실행한다. `llm`은 앞 결과에서 값 하나를 뽑는 일을 하위 모델에 맡긴다.
도구 이름은 `Literal`로 묶는다. 묶지 않으면 계획자가 `search_docs`처럼 없는 도구를 지어낸다.

In [12]:
class Node(BaseModel):
    id: str
    tool: Literal["grep_docs", "read_doc", "llm"]
    input: str


class Plan(BaseModel):
    nodes: list[Node]


PLANNER = ("다음 과제를 실행 계획으로 쓴다. 도구 결과를 보지 않고 미리 전부 쓴다.\n"
           "노드마다 id(E1, E2, ...), tool, input 을 정한다. tool 은 셋 중 하나다.\n"
           "- grep_docs: input 은 찾을 낱말이나 문서 제목. 결과는 문서 수와 문서별 제목, 등장 횟수 JSON\n"
           "- read_doc: input 은 문서 제목 하나. 결과는 본문 앞부분\n"
           "- llm: input 은 하위 모델에 줄 지시문. 앞 결과에서 값 하나를 뽑을 때 쓰고, 값만 답하라고 적는다\n"
           "input 에 #E1 처럼 앞 노드 id 를 쓰면 실행할 때 그 결과로 바뀐다."
           " 제목 하나가 필요한 자리에는 llm 노드로 뽑은 결과를 쓴다.\n"
           "마지막 표 정리는 노드에 넣지 않는다.\n\n")


def make_plan(task):
    return keep(client.responses.parse(model=MODEL, store=False, reasoning=EFFORT, text_format=Plan,
                                       input=PLANNER + task)).output_parsed


plan = make_plan(TASK)
print(len(plan.nodes), "노드")
for n in plan.nodes[:8]:
    print(f"  {n.id:4s} {n.tool:9s} {n.input[:90]}")

64 노드
  E1   grep_docs 독립신문
  E2   llm       #E1 결과 JSON에서 문서별 등장 횟수를 비교해 가장 많이 나온 문서의 제목 하나만 값으로 답하라.
  E3   grep_docs #E2
  E4   llm       #E3 결과 JSON에서 문서별 등장 횟수를 비교하되 제목이 #E2 인 문서는 제외하고, 남은 것 중 가장 많이 나온 문서의 제목 하나만 값으로 답하라.
  E5   grep_docs #E4
  E6   llm       #E5 결과 JSON에서 문서별 등장 횟수를 비교하되 제목이 #E2 또는 #E4 인 문서는 제외하고, 남은 것 중 가장 많이 나온 문서의 제목 하나만 값으로 답하
  E7   read_doc  #E6
  E8   llm       #E7 본문 앞부분에서 처음 나오는 연도를 찾아 연도 값만 답하라.


실행기는 자리표시자를 앞 결과로 바꾼 뒤 노드를 실행한다. 계획자를 다시 부르지 않는다.
계획자가 id 를 `E1` 꼴로 쓰지 않을 때도 있어서, `#` 뒤의 낱말이 계획에 있는 id 일 때만 바꾼다.

In [13]:
REF = re.compile(r"#(\w+)")


def run_node(node, ev):
    text = REF.sub(lambda m: ev.get(m[1], m[0]), node.input)
    if node.tool == "llm":
        return keep(client.responses.create(model=LOWER, store=False, reasoning=EFFORT,
                                            input=text)).output_text.strip()
    if node.tool == "grep_docs":
        return call_tool("grep_docs", {"pattern": text.strip()})
    if node.tool == "read_doc":
        return call_tool("read_doc", {"title": text.strip()})
    return f"error: 모르는 도구 {node.tool}"

모든 노드가 끝나면 풀이 단계가 실행 기록만 보고 표를 쓴다. 도구 결과는 앞 300자만 넘긴다. 뽑아 둔 값은 짧아서 다 들어간다.

In [14]:
def solve(task, plan, ev):
    lines = "\n".join(f"#{n.id} {n.tool}[{n.input}] = {ev[n.id][:300]}" for n in plan.nodes)
    return keep(client.responses.create(
        model=LOWER, store=False, reasoning=EFFORT,
        input=f"과제\n{task}\n\n실행 기록\n{lines}\n\n실행 기록의 값만 써서 과제의 표를 만든다.")).output_text


def rewoo(task, plan=None):
    plan = plan or make_plan(task)
    ev = {}
    for n in plan.nodes:
        ev[n.id] = run_node(n, ev)
    return solve(task, plan, ev), ev

위에서 세운 계획으로 한 번 돌린다. 첫 사슬의 실행 기록을 본다.

In [16]:
t0 = time.perf_counter()
answer, ev = rewoo(TASK, plan)
print(f"{time.perf_counter() - t0:.0f}초")
for n in plan.nodes[:5]:
    print(f"  #{n.id} = {ev[n.id][:70]}")
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

75초
  #E1 = {"문서수": 11, "문서": [{"title": "김구", "count": 1}, {"title": "김규식", "coun
  #E2 = 서재필
  #E3 = {"문서수": 12, "문서": [{"title": "김규식", "count": 17}, {"title": "남궁억", "co
  #E4 = 한국친우회
  #E5 = {"문서수": 3, "문서": [{"title": "서재필", "count": 9}, {"title": "이승만", "coun
32/32  못 채운 것 []


세 번 돌린다. 계획도 매번 다시 세운다.

In [17]:
RESULTS["ReWOO"] = trial("ReWOO", lambda: rewoo(TASK)[0])

ReWOO 1회  29/32  못 채운 것 ['신민회/2차', '신민회/3차', '신민회/연도']
ReWOO 2회  32/32  못 채운 것 []
ReWOO 3회  32/32  못 채운 것 []


### 의존 그래프로 실행한다 (DAG)

ReWOO 실행기는 노드를 순서대로 하나씩 돈다. 그런데 사슬끼리는 서로의 결과를 쓰지 않는다.
자리표시자가 곧 의존 관계이므로 계획을 그래프로 읽을 수 있다. **앞 노드가 모두 끝난 노드끼리 한 물결로 묶어 동시에 실행한다. (병렬적으로)**
DAG 는 Directed Acyclic Graph(방향 비순환 그래프)의 약자다. 화살표에 방향이 있고(`#E1` 을 쓰는 노드는 E1 다음에 온다),
화살표를 따라가도 출발한 노드로 되돌아오는 길이 없다. 순환이 없으니 실행 순서를 언제나 정할 수 있다.

In [18]:
def waves(plan):
    ids = {n.id for n in plan.nodes}
    deps = {n.id: set(REF.findall(n.input)) & ids for n in plan.nodes}
    done, out = set(), []
    while len(done) < len(plan.nodes):
        ready = [n for n in plan.nodes if n.id not in done and deps[n.id] <= done]
        if not ready:
            raise ValueError("순환 참조가 있거나 없는 노드를 가리킨다")
        out.append(ready)
        done |= {n.id for n in ready}
    return out


print("물결별 노드 수", [len(w) for w in waves(plan)])

물결별 노드 수 [8, 8, 8, 8, 8, 8, 8, 8]


노드 수는 그대로이고 물결 수만큼만 기다린다. 한 물결 안의 노드는 서로를 참조하지 않으므로 같은 `ev`를 읽어도 안전하다.

In [19]:
def dag(task, plan=None):
    plan = plan or make_plan(task)
    ev = {}
    with ThreadPoolExecutor(8) as ex:
        for wave in waves(plan):
            for n, result in zip(wave, ex.map(lambda n: run_node(n, ev), wave)):
                ev[n.id] = result
    return solve(task, plan, ev), ev


t0 = time.perf_counter()
answer, ev = dag(TASK, plan)
print(f"{time.perf_counter() - t0:.0f}초")
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

19초
32/32  못 채운 것 []


In [ ]:
RESULTS["DAG"] = trial("DAG", lambda: dag(TASK)[0])

### 관찰 없이 세운 계획은 빈 결과를 모른다

ReWOO 계획자는 도구 결과를 보지 않는다. 시작 낱말이 코퍼스에 없으면 첫 검색이 0건인데, 계획은 그대로 다음 노드로 넘어간다.
없는 낱말 하나로 사슬 하나짜리 계획을 세워 본다.

In [21]:
broken_task = TASK.split("\n- ")[0] + "\n- 조선민족혁명단\n마지막에 | 시작 | 1차 | 2차 | 3차 | 연도 | 다섯 칸짜리 마크다운 표 하나로 정리한다."
broken = make_plan(broken_task)
_, ev = dag(broken_task, broken)
for n in broken.nodes:
    print(f"  #{n.id} {n.tool:9s} = {ev[n.id][:70]}")

  #E1 grep_docs = {"문서수": 0, "문서": []}
  #E2 llm       = 없음
  #E3 grep_docs = {"문서수": 9, "문서": [{"title": "김구", "count": 1}, {"title": "남궁억", "count
  #E4 llm       = 서재필
  #E5 grep_docs = {"문서수": 12, "문서": [{"title": "김규식", "count": 17}, {"title": "남궁억", "co
  #E6 llm       = 한국친우회
  #E7 read_doc  = "# 한국친우회\n\n한국친우회(韓國親友會, The League of Friends of Korea)는 1919년 4월 서재필
  #E8 llm       = 1919


`llm` 노드가 0건 결과에서 "없음" 같은 답을 내도 그 답이 그대로 다음 검색어가 된다. 엉뚱한 사슬이 끝까지 이어지고 표에는 그럴듯한 값이 들어간다.
계획이 미리 쓰여 있으니 실행기가 결과를 검사해야 한다. 여기서는 검색이 0건이면 그 노드를 실패로 표시하고, 실패한 노드를 참조하는 노드는 실행하지 않는다.
실패가 남은 사슬은 `4`의 재계획처럼 그 부분만 계획자에게 돌려보내 다시 세운다.

In [22]:
def run_node_checked(node, ev):
    if any(ev.get(r, "").startswith("실패") for r in REF.findall(node.input)):
        return "실패: 앞 노드가 실패했다"
    result = run_node(node, ev)
    if node.tool == "grep_docs" and json.loads(result).get("문서수") == 0:
        return f"실패: '{node.input}' 검색 결과 0건"
    return result


ev = {}
for wave in waves(broken):
    for n in wave:
        ev[n.id] = run_node_checked(n, ev)
for n in broken.nodes:
    print(f"  #{n.id} = {ev[n.id][:70]}")

  #E1 = 실패: '조선민족혁명단' 검색 결과 0건
  #E2 = 실패: 앞 노드가 실패했다
  #E3 = 실패: 앞 노드가 실패했다
  #E4 = 실패: 앞 노드가 실패했다
  #E5 = 실패: 앞 노드가 실패했다
  #E6 = 실패: 앞 노드가 실패했다
  #E7 = 실패: 앞 노드가 실패했다
  #E8 = 실패: 앞 노드가 실패했다


### 나란히 둔다

ReWOO 와 DAG 는 계획만 상위 모델이 세우고 나머지는 하위 모델이 한다.

In [ ]:
base = RESULTS["ReAct(상위)"]["비용"]
for label, r in RESULTS.items():
    print(f"{label:18s} {str(r['점수']):14s} 평균 {sum(r['점수']) / len(r['점수']):4.1f}/{FULL}"
          f"   호출 {r['호출']:3d}회   ${r['비용']:.3f}  (상위 ReAct 의 {r['비용'] / base:4.0%})"
          f"   {r['초'] / len(r['점수']):4.0f}초/회")

### 정리

하위 모델 ReAct 는 사슬 여러 개를 한 대화에서 따라가다가 한 단계를 건너뛰거나 다른 사슬의 값을 가져온다. effort 를 올리면 나아지지만 사슬 하나가 통째로 틀리는 경우가 남고, 비용과 시간은 상위 모델 ReAct 에 가깝게 는다.

자리표시자 계획은 "다음 검색어를 모른다"는 문제를 값 대신 참조로 푼다. 계획자는 한 번만 부르고, 실행기는 노드마다 작은 일 하나만 맡으므로 하위 모델로도 상위 모델 ReAct 와 같은 수준의 점수가 나온다. 호출 수는 몇 배로 늘고 비용은 조금 준다.
순서대로 실행하면 상위 모델 ReAct 보다 느리다. 같은 계획을 의존 그래프로 읽으면 노드 수와 비용은 그대로이고 기다리는 시간이 물결 수만큼으로 준다.
계획자가 계획 형식을 어기면 실행 전체가 무너진다. 도구 이름을 `Literal`로 묶은 것이 그 방어다.

대가는 계획이 관찰을 보지 않는다는 것이다. 빈 결과나 예상 밖의 값이 나와도 계획은 그대로 진행되므로, 실행기가 결과를 검사하고 실패를 전파하고 필요한 부분만 다시 세워야 한다.
다음 단계가 앞 결과의 값뿐 아니라 종류에 따라 달라지는 과제, 예를 들어 결과를 보고 어떤 도구를 쓸지 정해야 하는 과제는 자리표시자로 쓸 수 없다. 그때는 ReAct 나 단계마다 재계획하는 쪽이 맞다.

### 비용 확인

In [ ]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")